In [0]:
# =============================================================
# silver_writer.py
# Layer     : Silver
# Purpose   : Delta writes only.
#             - MERGE (upsert) for incremental loads
#             - Overwrite for first run and dimensions
#             - Audit column injection
#
# STRICT RULE: No reads. No cleaning. No schema definitions.
#              If you see spark.read here → wrong file.
# =============================================================

from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable


# ── Primary key registry ──────────────────────────────────────────────────────
# Drives the MERGE ON condition per table.
# Composite keys for snapshot tables (device + timestamp = unique row).

_PK_MAP = {
    "owners":               ["owner_id"],
    "pets":                 ["pet_id"],
    "products":             ["product_id"],
    "firmware":             ["firmware_id"],
    "batches":              ["batch_id"],
    "devices":              ["device_id"],
    "sales":                ["sale_id"],
    "device_snapshots":     ["device_id", "snapshot_timestamp"],
    "activity_snapshots":   ["device_id", "snapshot_timestamp"],
    "health_snapshots":     ["device_id", "snapshot_timestamp"],
    "feeding_events":       ["feed_event_id"],
    "hydration_events":     ["water_event_id"],
    "device_failures":      ["failure_id"],
    "firmware_deployments": ["deployment_id"],
}


def add_silver_audit(df):
    """
    Add Silver-layer audit column.
    Kept separate from Bronze lineage — Silver owns its own timestamp.
    """
    return df.withColumn("_silver_load_timestamp", current_timestamp())


def write_silver(df, table_name: str, is_incremental: bool) -> int:
    """
    Write cleaned DataFrame to Silver Delta.

    Strategy:
        First run OR dimension table → full overwrite
            (DeltaTable.isDeltaTable returns False on first run)
        Subsequent incremental fact runs → MERGE (upsert)
            Matched rows: update all columns
            New rows: insert

    Args:
        df             : Cleaned, typed DataFrame with _silver_load_timestamp
        table_name     : e.g. "device_snapshots"
        is_incremental : from ADF widget

    Returns:
        Final Silver row count
    """
    silver_path = abfss("silver", table_name)

    if table_name not in _PK_MAP:
        raise KeyError(
            f"[silver_writer] No primary key defined for '{table_name}'. "
            f"Add it to _PK_MAP."
        )

    pk_cols = _PK_MAP[table_name]

    if is_incremental and DeltaTable.isDeltaTable(spark, silver_path):
        # ── MERGE (upsert) ────────────────────────────────────────────────────
        merge_condition = " AND ".join(
            [f"target.{k} = source.{k}" for k in pk_cols]
        )
        print(f"[silver_writer] MERGE into: {silver_path}")
        print(f"[silver_writer] ON: {merge_condition}")

        silver_table = DeltaTable.forPath(spark, silver_path)
        (silver_table.alias("target")
            .merge(df.alias("source"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"[silver_writer] MERGE complete.")

    else:
        # ── Full overwrite ─────────────────────────────────────────────────────
        print(f"[silver_writer] Overwrite: {silver_path}")
        (df.write
           .format("delta")
           .mode("overwrite")
           .option("overwriteSchema", "true")
           .save(silver_path)
        )
        print(f"[silver_writer] Overwrite complete.")

    # Always return final count from Silver
    final_count = spark.read.format("delta").load(silver_path).count()
    print(f"[silver_writer] Silver rows after write: {final_count:,}")
    return final_count

print("[silver_writer] Loaded.")